In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) Row counts per file
row_counts = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'row_count': [metric_df.shape[0], trace_df.shape[0], log_df.shape[0], error_df.shape[0]]
})

# 2) Unique cmdb_id values per file (list up to 50 values)
def _cmdb_list(df):
    vals = pd.Series(df['cmdb_id'].dropna().unique()).astype(str)
    top_vals = vals.head(50).tolist()
    return ";".join(top_vals)

unique_cmdbs_df = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'unique_cmdb_ids_sample': [
        _cmdb_list(metric_df),
        _cmdb_list(trace_df),
        _cmdb_list(log_df),
        _cmdb_list(error_df)
    ],
    'unique_count': [
        metric_df['cmdb_id'].nunique(dropna=True),
        trace_df['cmdb_id'].nunique(dropna=True),
        log_df['cmdb_id'].nunique(dropna=True),
        error_df['cmdb_id'].nunique(dropna=True)
    ]
})

# 3) metric.csv: unique kpi_name with counts, sorted desc (top 20)
metric_kpi_counts = (
    metric_df.groupby('kpi_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# 4) trace.csv: unique trace_name with counts, sorted desc (top 20)
trace_name_counts = (
    trace_df.groupby('trace_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# 5) log.csv: unique log_name with counts, sorted desc (top 20)
log_name_counts = (
    log_df.groupby('log_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# 6) metric.csv: global 95th percentile (P95) of 'value' per (cmdb_id, kpi_name) across entire file
# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

metric_p95 = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
    .sort_values('p95_value', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# Display compact, machine-readable summaries (CSV-like / small tables)
row_counts, unique_cmdbs_df, metric_kpi_counts, trace_name_counts, log_name_counts, metric_p95

```
Out[1]:
```
```python
# Summarize the findings in plain English (compact)
summary = (
    "Summary of telemetry aggregates:\n"
    "- Row counts: metric=1875, trace=3728, log=454, error_logs=0.\n"
    "- Unique CMDB IDs: metric=12, trace=8, log=10, error_logs=0.\n"
    "- Top metric KPI names (by count): cpu, workload, socket, mem (each 275 occurrences); "
    "followed by latency-50 and latency-90 (250), diskio (225), error (50).\n"
    "- Top trace features: trace.from_root.* (172 rows each), trace.from_frontendservice.* (124 rows each), "
    "trace.from_checkoutservice.* (120 rows each), and traces to productcatalogservice and recommendationservice.\n"
    "- Log features: log.error_count and log.row_count (each 227 rows).\n"
    "- Metric P95 highlights (top by p95_value):\n"
    "  * adservice mem P95 ≈ 1.1032e+08 (highest)\n"
    "  * currencyservice mem P95 ≈ 5.0040e+07\n"
    "  * recommendationservice mem P95 ≈ 4.5635e+07\n"
    "  * paymentservice mem P95 ≈ 4.5071e+07\n"
    "  * cartservice mem P95 ≈ 4.4815e+07\n"
    "  * adservice diskio P95 ≈ 7.4269e+04 (notable)\n"
    "  * other notable P95s include workload and cpu on several services.\n\n"
    "Interpretation / straightforward conclusion:\n"
    "- Memory-related metrics dominate the high P95 values, with adservice showing the largest memory P95 by far. "
    "This points to high memory usage (especially in adservice) as the most prominent signal in the telemetry. "
    "- There are no entries in error_logs to add contextual error messages. Traces and logs are present and show "
    "common trace groups (from_root, from_frontendservice, from_checkoutservice) and log.error_count activity.\n"
    "- Primary area to investigate first: adservice memory usage, then memory of currency/recommendation/payment/cart services; "
    "also check adservice disk I/O as a secondary concern."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count
0      metric.csv       1875
1       trace.csv       3728
2         log.csv        454
3  error_logs.csv          0,              file                             unique_cmdb_ids_sample  unique_count
0      metric.csv  adservice;cartservice;checkoutservice;currency...            12
1       trace.csv  currencyservice;emailservice;frontendservice;p...             8
2         log.csv  adservice;cartservice;currencyservice;frontend...            10
3  error_logs.csv                                                                0,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    225
7       error     50,                                       trace_name  count
0                      trace.from_root.row_count    172
1                     trace.from_root.error_rate    172
2                   trace.from_root.duration_p95    172
3                  trace.from_root.duration_mean    172
4           trace.from_frontendservice.row_count    124
5          trace.from_frontendservice.error_rate    124
6        trace.from_frontendservice.duration_p95    124
7       trace.from_frontendservice.duration_mean    124
8           trace.from_checkoutservice.row_count    120
9          trace.from_checkoutservice.error_rate    120
10       trace.from_checkoutservice.duration_p95    120
11      trace.from_checkoutservice.duration_mean    120
12  trace.to_productcatalogservice.duration_mean     99
13   trace.to_productcatalogservice.duration_p95     99
14     trace.to_productcatalogservice.error_rate     99
15      trace.to_productcatalogservice.row_count     99
16  trace.to_recommendationservice.duration_mean     75
17   trace.to_recommendationservice.duration_p95     75
18     trace.to_recommendationservice.error_rate     75
19      trace.to_recommendationservice.row_count     75,           log_name  count
0  log.error_count    227
1    log.row_count    227,                   cmdb_id  kpi_name     p95_value
0               adservice       mem  1.103221e+08
1         currencyservice       mem  5.004042e+07
2   recommendationservice       mem  4.563542e+07
3          paymentservice       mem  4.507069e+07
4             cartservice       mem  4.481460e+07
5            emailservice       mem  4.396323e+07
6                frontend       mem  1.664153e+07
7         checkoutservice       mem  1.193748e+07
8   productcatalogservice       mem  1.089337e+07
9         shippingservice       mem  9.135814e+06
10                  redis       mem  2.993698e+06
11              adservice    diskio  7.426916e+04
12                  redis    diskio  8.589863e+03
13  productcatalogservice  workload  1.492196e+02
14        currencyservice  workload  8.208053e+01
15        currencyservice       cpu  4.975862e+01
16               frontend    socket  2.700000e+01
17            cartservice  workload  2.668541e+01
18  recommendationservice  workload  1.960581e+01
19              adservice  workload  1.567331e+01)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df if present; otherwise load and parse
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) across the entire metric file (do NOT filter by time)
metric_global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# 2) Define incident window (UTC)
start = pd.Timestamp("2024-01-19 17:51:58", tz='UTC')
end   = pd.Timestamp("2024-01-19 18:21:58", tz='UTC')

# 3) Filter metric_df to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 4) Merge global p95 into the window data
metric_window = metric_window.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 5) Mark anomaly rows where value >= global p95 (handle NaN p95 by treating as False)
metric_window['is_anomaly'] = (metric_window['value'] >= metric_window['p95_value']) & metric_window['p95_value'].notna()

# 6) Aggregate per (cmdb_id, kpi_name) within the incident window
grouped = metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False)

agg_basic = grouped.agg(
    anomaly_count = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    p95_value = ('p95_value', 'first'),
    total_points_in_window = ('value', 'count')
).reset_index()

# 7) Earliest anomaly timestamp per group (from anomaly rows)
earliest = (
    metric_window[metric_window['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# 8) Merge earliest into agg_basic
agg = agg_basic.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# 9) Keep only rows with anomaly_count > 0
agg = agg[agg['anomaly_count'] > 0].copy()

# 10) Compute severity ratio for sorting: max_value_in_window / p95_value (handle NaN or zero)
agg['severity_ratio'] = np.where(
    (agg['p95_value'].notna()) & (agg['p95_value'] != 0),
    agg['max_value_in_window'] / agg['p95_value'],
    np.inf
)

# 11) Sort by anomaly_count desc, then severity_ratio desc
agg = agg.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 12) Prepare final compact CSV-like table with requested columns and formatting, limit to top 20
result_df = agg.loc[:, ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','total_points_in_window']].head(20)

# Format earliest_anomaly_timestamp as ISO UTC strings for compact display
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp'].astype(str)

# Display final result (compact)
result_df

```
Out[2]:
```
Summary of anomalies inside the incident window (2024-01-19 17:51:58 → 2024-01-19 18:21:58 UTC):

- General:
  - Each series in the window had 25 points (total_points_in_window = 25), i.e., 25 minutes of data.
  - Many series show anomaly_count = 25 (every point >= global P95). This is often because the global P95 for those KPIs is 0, so equality to 0 marks all points as anomalies — treat those as potential false positives and verify whether a P95 of 0 is meaningful for that KPI.

- Top anomaly rows (highest anomaly_count first; earliest anomaly timestamp shown):
  1. cartservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  2. currencyservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  3. emailservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  4. frontend, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  5. frontend, error — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  6. frontend-external, error — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.03325, p95=0.0
  7. paymentservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  8. productcatalogservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
  9. recommendationservice, diskio — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=0.0, p95=0.0
 10. adservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=4.0, p95=4.0
 11. cartservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=4.0, p95=4.0
 12. checkoutservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=9.0, p95=9.0
 13. emailservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=3.0, p95=3.0
 14. paymentservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=3.0, p95=3.0
 15. productcatalogservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=3.0, p95=3.0
 16. recommendationservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=4.0, p95=4.0
 17. redis, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=4.0, p95=4.0
 18. shippingservice, socket — anomaly_count=25, earliest=2024-01-19T17:54:00Z, max=2.0, p95=2.0
 19. currencyservice, socket — anomaly_count=11, earliest=2024-01-19T18:08:00Z, max=6.0, p95=6.0
 20. frontend, socket — anomaly_count=5, earliest=2024-01-19T18:14:00Z, max=27.0, p95=27.0

- Interpretation and recommended next steps:
  - Many diskio and error KPIs have global P95 = 0; the result that all window points equal or exceed P95 is likely not a true anomaly signal. Confirm whether those KPIs are sparse/mostly-zero by nature; consider using a nonzero baseline or additional context before acting on them.
  - Socket-related KPIs are notable and more credible here because p95 values are nonzero and max values match p95 (indicating sustained high socket counts during the window). In particular, frontend socket max=27 (anomaly_count=5) and checkoutservice socket max=9 (anomaly_count=25) merit investigation.
  - Prioritize investigating socket issues (frontend, checkoutservice, adservice, etc.). Treat diskio/error rows with p95=0 cautiously and verify telemetry semantics before escalating.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  total_points_in_window
0             cartservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
1         currencyservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
2            emailservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
3                frontend   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
4                frontend    error             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
5       frontend-external    error             25  2024-01-19 17:54:00+00:00              0.03325        0.0                      25
6          paymentservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
7   productcatalogservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
8   recommendationservice   diskio             25  2024-01-19 17:54:00+00:00              0.00000        0.0                      25
9               adservice   socket             25  2024-01-19 17:54:00+00:00              4.00000        4.0                      25
10            cartservice   socket             25  2024-01-19 17:54:00+00:00              4.00000        4.0                      25
11        checkoutservice   socket             25  2024-01-19 17:54:00+00:00              9.00000        9.0                      25
12           emailservice   socket             25  2024-01-19 17:54:00+00:00              3.00000        3.0                      25
13         paymentservice   socket             25  2024-01-19 17:54:00+00:00              3.00000        3.0                      25
14  productcatalogservice   socket             25  2024-01-19 17:54:00+00:00              3.00000        3.0                      25
15  recommendationservice   socket             25  2024-01-19 17:54:00+00:00              4.00000        4.0                      25
16                  redis   socket             25  2024-01-19 17:54:00+00:00              4.00000        4.0                      25
17        shippingservice   socket             25  2024-01-19 17:54:00+00:00              2.00000        2.0                      25
18        currencyservice   socket             11  2024-01-19 18:08:00+00:00              6.00000        6.0                      25
19               frontend   socket              5  2024-01-19 18:14:00+00:00             27.00000       27.0                      25```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Global P95 per (cmdb_id, trace_name) across entire trace.csv (no time filter)
trace_global_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# 2) Incident window (UTC)
start = pd.Timestamp("2024-01-19 17:51:58", tz='UTC')
end   = pd.Timestamp("2024-01-19 18:21:58", tz='UTC')

# 3) Filter trace_df to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 4) Merge global p95 into the window data
trace_window = trace_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# 5) Mark anomaly rows where value >= global p95 (only when p95_value is not null)
trace_window['is_anomaly'] = (trace_window['p95_value'].notna()) & (trace_window['value'] >= trace_window['p95_value'])

# 6) Aggregate per (cmdb_id, trace_name) within the incident window
grouped = trace_window.groupby(['cmdb_id', 'trace_name'], dropna=False)

agg_basic = grouped.agg(
    anomaly_count = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    p95_value = ('p95_value', 'first'),
    total_points_in_window = ('value', 'count')
).reset_index()

# 7) Earliest anomaly timestamp per group
earliest = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_utc')
)

# 8) Merge earliest into agg_basic
agg = agg_basic.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# 9) Keep only rows with anomaly_count > 0
agg = agg[agg['anomaly_count'] > 0].copy()

# 10) Compute severity ratio for sorting: max_value_in_window / p95_value (handle NaN/zero)
agg['severity_ratio'] = np.where(
    (agg['p95_value'].notna()) & (agg['p95_value'] != 0),
    agg['max_value_in_window'] / agg['p95_value'],
    np.inf
)

# 11) Sort by anomaly_count desc, then severity_ratio desc
agg = agg.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 12) Prepare final compact CSV-like table (top 20) with requested columns
trace_anomalies_result = agg.loc[:, [
    'cmdb_id',
    'trace_name',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'p95_value',
    'total_points_in_window'
]].head(20).copy()

# Format earliest_anomaly_timestamp_utc as ISO UTC strings for compact display
trace_anomalies_result['earliest_anomaly_timestamp_utc'] = trace_anomalies_result['earliest_anomaly_timestamp_utc'].astype(str)

# Display final result (compact)
trace_anomalies_result

```
Out[3]:
```
```python
# Plain-English summary as a string (compact)
summary = (
    "Trace analysis summary (incident window 2024-01-19 17:51:58 → 2024-01-19 18:21:58 UTC):\n"
    "- All reported trace anomalies are for error_rate metrics (many trace_name entries) and each group shows\n"
    "  anomaly_count = 25 with p95_value = 0.0 and max_value_in_window = 0.0.\n"
    "- Because the global P95 for these trace series is 0, every non-null point (>= 0) in the window is flagged as\n"
    "  an anomaly. This is likely a false-positive signal caused by a zero baseline (sparse/mostly-zero error_rate\n"
    "  telemetry), not a real spike in errors.\n"
    "- There are no credible non-zero trace anomalies in the window to escalate from this result.\n"
    "- Recommended next steps: verify the semantics and sparsity of trace.error_rate (why P95==0), cross-check with\n"
    "  logs/metrics for error events, and focus investigation on signals with non-zero P95 (e.g., socket/memory from\n"
    "  metric analysis) rather than these zero-baseline trace error_rate series."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window  p95_value  total_points_in_window
0         currencyservice        trace.from_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
1            emailservice                   trace.from_root.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
2         frontendservice        trace.from_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
3         frontendservice                   trace.from_root.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
4         frontendservice          trace.to_currencyservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
5         frontendservice          trace.to_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
6         frontendservice    trace.to_productcatalogservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
7         frontendservice    trace.to_recommendationservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
8   productcatalogservice        trace.from_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
9   productcatalogservice  trace.from_recommendationservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
10  productcatalogservice                   trace.from_root.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
11  recommendationservice        trace.from_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
12  recommendationservice  trace.from_recommendationservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
13  recommendationservice                   trace.from_root.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
14  recommendationservice    trace.to_productcatalogservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
15  recommendationservice    trace.to_recommendationservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
16                   root             trace.to_emailservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
17                   root          trace.to_frontendservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
18                   root    trace.to_productcatalogservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25
19                   root    trace.to_recommendationservice.error_rate             25      2024-01-19 17:54:00+00:00                  0.0        0.0                      25```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Select trace_name values that contain 'duration' and compute global P95 per (cmdb_id, trace_name)
is_duration = trace_df['trace_name'].astype(str).str.contains('duration', na=False)
trace_duration_all = trace_df[is_duration].copy()

trace_duration_p95 = (
    trace_duration_all
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# 2) Incident window (UTC)
start = pd.Timestamp("2024-01-19 17:51:58", tz='UTC')
end   = pd.Timestamp("2024-01-19 18:21:58", tz='UTC')

# 3) Filter trace_df to the incident window and to duration trace_names
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
trace_window = trace_window[trace_window['trace_name'].astype(str).str.contains('duration', na=False)].copy()

# 4) Merge global p95 into the window data (p95 computed from entire file)
trace_window = trace_window.merge(trace_duration_p95, on=['cmdb_id', 'trace_name'], how='left')

# 5) Mark anomaly rows where value >= global p95 (only when p95_value is not null)
trace_window['is_anomaly'] = (trace_window['p95_value'].notna()) & (trace_window['value'] >= trace_window['p95_value'])

# 6) Aggregate per (cmdb_id, trace_name) within the incident window
grouped = trace_window.groupby(['cmdb_id', 'trace_name'], dropna=False)

agg_basic = grouped.agg(
    anomaly_count = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    p95_value = ('p95_value', 'first'),
    total_points_in_window = ('value', 'count')
).reset_index()

# 7) Earliest anomaly timestamp per group
earliest = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_utc')
)

# 8) Merge earliest into agg_basic
agg = agg_basic.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# 9) Keep only rows with anomaly_count > 0
agg = agg[agg['anomaly_count'] > 0].copy()

# 10) Compute severity ratio for sorting: max_value_in_window / p95_value (handle NaN/zero)
agg['severity_ratio'] = np.where(
    (agg['p95_value'].notna()) & (agg['p95_value'] != 0),
    agg['max_value_in_window'] / agg['p95_value'],
    np.inf
)

# 11) Sort by anomaly_count desc, then severity_ratio desc
agg = agg.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 12) Prepare final compact CSV-like table with requested columns (top 20)
trace_duration_anomalies_result = agg.loc[:, [
    'cmdb_id',
    'trace_name',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'p95_value',
    'total_points_in_window'
]].head(20).copy()

# Format earliest_anomaly_timestamp_utc as ISO UTC strings for compact display
trace_duration_anomalies_result['earliest_anomaly_timestamp_utc'] = trace_duration_anomalies_result['earliest_anomaly_timestamp_utc'].astype(str)

# Display final result (compact)
trace_duration_anomalies_result

```
Out[4]:
```
```python
# Plain-English summary of trace duration anomalies (compact)
summary = (
    "Trace duration anomaly summary (incident window 2024-01-19 17:51:58 → 2024-01-19 18:21:58 UTC):\n"
    "- There are a small number of duration anomalies per series: each reported (cmdb_id,trace_name) had anomaly_count = 2\n"
    "  (i.e., two points in the window exceeded the global P95 for that series).\n"
    "- The affected operations are distributed across services and spans: email-related (emailservice, root→emailservice),\n"
    "  payment/checkout/currency interactions, productcatalog↔recommendation traces, and frontend→checkout interactions.\n"
    "- Absolute durations for most anomalies are very small (sub-millisecond to a few milliseconds), but the most notable\n"
    "  latency signal is in frontend-related traces: frontend trace max ≈ 0.692s with global P95 ≈ 0.582s (root→frontend and\n"
    "  frontend→checkout entries at 18:10 UTC). This is the largest relative and absolute duration increase in the table.\n"
    "- Other duration anomalies (e.g., checkout→currency, payment/from_root, productcatalog/recommendation) have max values\n"
    "  in the sub-millisecond to low-millisecond range and smaller relative increases above their P95s.\n"
    "- Interpretation and next steps:\n"
    "  1) Prioritize investigating frontend latency around ~18:10 UTC (trace.from_root.duration_p95 / duration_mean entries),\n"
    "     since they show the largest absolute durations and exceed their global P95 by the largest margin.\n"
    "  2) Inspect traces for the checkout/payment/currency/email flows for the two anomalous timestamps (around 17:54, 17:57–18:08,\n"
    "     and 18:10) to find slow spans and upstream dependencies.\n"
    "  3) Correlate these duration anomalies with metric signals (CPU, memory, socket, disk I/O) and logs to determine\n"
    "     whether the latency is caused by resource pressure, downstream service latency, or transient errors.\n"
    "  4) Because anomaly_count is small (2 points), treat these as brief spikes; confirm whether they correlate with user\n"
    "     traffic or deployment events before escalating.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     trace_name  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window  p95_value  total_points_in_window
0            emailservice                   trace.from_root.duration_p95              2      2024-01-19 18:03:00+00:00             0.000219   0.000128                      25
1                    root             trace.to_emailservice.duration_p95              2      2024-01-19 18:03:00+00:00             0.000219   0.000128                      25
2         checkoutservice          trace.to_currencyservice.duration_p95              2      2024-01-19 18:08:00+00:00             0.000822   0.000543                      24
3         currencyservice        trace.from_checkoutservice.duration_p95              2      2024-01-19 18:08:00+00:00             0.000822   0.000543                      24
4          paymentservice                   trace.from_root.duration_p95              2      2024-01-19 17:58:00+00:00             0.000237   0.000169                      24
5                    root           trace.to_paymentservice.duration_p95              2      2024-01-19 17:58:00+00:00             0.000237   0.000169                      24
6   productcatalogservice  trace.from_recommendationservice.duration_p95              2      2024-01-19 17:54:00+00:00             0.000041   0.000031                      25
7   recommendationservice    trace.to_productcatalogservice.duration_p95              2      2024-01-19 17:54:00+00:00             0.000041   0.000031                      25
8          paymentservice                  trace.from_root.duration_mean              2      2024-01-19 17:58:00+00:00             0.000099   0.000077                      24
9                    root          trace.to_paymentservice.duration_mean              2      2024-01-19 17:58:00+00:00             0.000099   0.000077                      24
10        checkoutservice          trace.to_paymentservice.duration_mean              2      2024-01-19 17:57:00+00:00             0.000456   0.000364                      24
11         paymentservice       trace.from_checkoutservice.duration_mean              2      2024-01-19 17:57:00+00:00             0.000456   0.000364                      24
12        checkoutservice             trace.to_emailservice.duration_p95              2      2024-01-19 17:57:00+00:00             0.000675   0.000556                      24
13           emailservice        trace.from_checkoutservice.duration_p95              2      2024-01-19 17:57:00+00:00             0.000675   0.000556                      24
14        currencyservice                   trace.from_root.duration_p95              2      2024-01-19 18:14:00+00:00             0.026739   0.022347                      24
15                   root          trace.to_currencyservice.duration_p95              2      2024-01-19 18:14:00+00:00             0.026739   0.022347                      24
16        checkoutservice       trace.from_frontendservice.duration_mean              2      2024-01-19 18:10:00+00:00             0.344809   0.289717                      24
17        frontendservice         trace.to_checkoutservice.duration_mean              2      2024-01-19 18:10:00+00:00             0.344809   0.289717                      24
18        frontendservice                   trace.from_root.duration_p95              2      2024-01-19 18:10:00+00:00             0.691981   0.582320                      25
19                   root          trace.to_frontendservice.duration_p95              2      2024-01-19 18:10:00+00:00             0.691981   0.582320                      25```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing variables if present; otherwise load files
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes and ensure numeric 'value'
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) Global P95 per (cmdb_id, log_name) across entire log.csv (no time filter)
log_global_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# 2) Incident window (UTC)
start = pd.Timestamp("2024-01-19 17:51:58", tz='UTC')
end   = pd.Timestamp("2024-01-19 18:21:58", tz='UTC')

# 3) Filter log_df to incident window
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# Merge global p95 into window data
log_window = log_window.merge(log_global_p95, on=['cmdb_id', 'log_name'], how='left')

# Mark anomaly rows where value >= global p95 (only when p95_value is not null)
log_window['is_anomaly'] = (log_window['p95_value'].notna()) & (log_window['value'] >= log_window['p95_value'])

# Aggregate per (cmdb_id, log_name) within the window
grouped = log_window.groupby(['cmdb_id', 'log_name'], dropna=False)

agg_basic = grouped.agg(
    anomaly_count = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    p95_value = ('p95_value', 'first'),
    total_points_in_window = ('value', 'count')
).reset_index()

# Earliest anomaly timestamp per group
earliest = (
    log_window[log_window['is_anomaly']]
    .groupby(['cmdb_id','log_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_utc')
)

# Merge earliest into agg_basic
agg = agg_basic.merge(earliest, on=['cmdb_id','log_name'], how='left')

# Keep only rows with anomaly_count > 0
agg = agg[agg['anomaly_count'] > 0].copy()

# Compute severity ratio for sorting: max_value_in_window / p95_value (handle NaN/zero)
agg['severity_ratio'] = np.where(
    (agg['p95_value'].notna()) & (agg['p95_value'] != 0),
    agg['max_value_in_window'] / agg['p95_value'],
    np.inf
)

# Sort by anomaly_count desc, then severity_ratio desc
agg = agg.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# Prepare final compact CSV-like table with requested columns (top 20)
log_anomalies_result = agg.loc[:, [
    'cmdb_id',
    'log_name',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'p95_value',
    'total_points_in_window'
]].head(20).copy()

# Format earliest_anomaly_timestamp_utc as ISO UTC strings for compact display
log_anomalies_result['earliest_anomaly_timestamp_utc'] = log_anomalies_result['earliest_anomaly_timestamp_utc'].astype(str)

# Error logs single-line summary: row count and up to 5 sample rows (timestamp_utc,cmdb_id,message)
error_count = error_df.shape[0]
if error_count == 0:
    error_summary = f"error_logs.csv rows=0"
else:
    samples = error_df[['timestamp','cmdb_id','message']].head(5).copy()
    samples['timestamp'] = samples['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    sample_strs = samples.apply(lambda r: f"{r['timestamp']},{r['cmdb_id']},{r['message']}", axis=1).tolist()
    error_summary = f"error_logs.csv rows={error_count}; samples: " + " ; ".join(sample_strs)

# Display results compactly: log anomalies table and single-line error summary
log_anomalies_result, error_summary

```
Out[5]:
```
Summary of log analysis (incident window 2024-01-19 17:51:58 → 2024-01-19 18:21:58 UTC):

- error_logs.csv: 0 rows.

- Global-P95 behavior:
  - Many log.error_count series have global P95 = 0, so every zero/non-null point in the window is flagged (anomaly_count=24–25). These are likely false positives due to a zero baseline and should be treated cautiously.
  - log.row_count series have nonzero P95s and show meaningful spikes.

- Key findings (compact):
  - log.error_count: multiple services (adservice, cartservice, currencyservice, frontend, recommendationservice, shippingservice, etc.) show anomaly_count = 24–25 but max_value_in_window = 0 and p95_value = 0 → likely not actionable (zero-baseline).
  - log.row_count (more credible signals):
    - frontend: max=2823, p95≈2803.8, anomaly_count=2, earliest=2024-01-19T17:59:00Z
    - currencyservice: max=2556, p95≈2537.6, anomaly_count=2, earliest=2024-01-19T17:59:00Z
    - cartservice: max=825, p95≈807.0, anomaly_count=2, earliest=2024-01-19T17:58:00Z
    - shippingservice: max=538, p95≈514.8, anomaly_count=2, earliest=2024-01-19T17:58:00Z
    - recommendationservice: max=594, p95≈588.0, anomaly_count=2, earliest=2024-01-19T17:56:00Z
    - adservice: max=471, p95≈466.4, anomaly_count=2, earliest=2024-01-19T17:59:00Z
    - checkout/payment/email: smaller but similar two-point spikes with max > p95 (earliest ≈ 17:56–17:58 UTC)
    - redis: log.row_count shows 5 points in-window (max=5, p95=5) and log.error_count with 5 anomalies (max=0, p95=0).

- Interpretation and recommended next steps:
  1. Ignore/verify log.error_count anomalies driven by global P95=0 before acting — these are likely artifacts of sparse/mostly-zero error_count telemetry.
  2. Investigate the log.row_count spikes (frontend, currencyservice, cartservice, shippingservice, recommendationservice, adservice, etc.) around 17:56–17:59 UTC — these are small but real increases above their P95 and may reflect traffic bursts or processing backlogs.
  3. Correlate these row_count spikes with the previously observed metric and trace signals (socket, duration spikes around 18:10, and the duration anomalies) and with deployment or traffic changes to determine root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window  p95_value  total_points_in_window
0               adservice  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
1             cartservice  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
2         currencyservice  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
3                frontend  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
4   recommendationservice  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
5         shippingservice  log.error_count             25      2024-01-19 17:54:00+00:00                    0       0.00                      25
6         checkoutservice  log.error_count             24      2024-01-19 17:55:00+00:00                    0       0.00                      24
7            emailservice  log.error_count             24      2024-01-19 17:55:00+00:00                    0       0.00                      24
8          paymentservice  log.error_count             24      2024-01-19 17:55:00+00:00                    0       0.00                      24
9                   redis  log.error_count              5      2024-01-19 17:58:00+00:00                    0       0.00                       5
10                  redis    log.row_count              5      2024-01-19 17:58:00+00:00                    5       5.00                       5
11           emailservice    log.row_count              2      2024-01-19 17:56:00+00:00                   39      34.85                      24
12         paymentservice    log.row_count              2      2024-01-19 17:56:00+00:00                   78      69.70                      24
13        checkoutservice    log.row_count              2      2024-01-19 17:56:00+00:00                  117     104.70                      24
14        shippingservice    log.row_count              2      2024-01-19 17:58:00+00:00                  538     514.80                      25
15            cartservice    log.row_count              2      2024-01-19 17:58:00+00:00                  825     807.00                      25
16  recommendationservice    log.row_count              2      2024-01-19 17:56:00+00:00                  594     588.00                      25
17              adservice    log.row_count              2      2024-01-19 17:59:00+00:00                  471     466.40                      25
18        currencyservice    log.row_count              2      2024-01-19 17:59:00+00:00                 2556    2537.60                      25
19               frontend    log.row_count              2      2024-01-19 17:59:00+00:00                 2823    2803.80                      25, 'error_logs.csv rows=0')```
```